## LangGraph를 위한 AgentCore Evaluations 온디맨드 평가

이 튜토리얼에서는 LangGraph 에이전트에 AgentCore Evaluations의 온디맨드 평가를 적용하는 방법을 알아봅니다.

이 실습을 진행하려면 먼저 [00-prereqs](../../00-prereqs) 폴더의 코드로 LangGraph 에이전트를 생성하고, [01-creating-custom-evaluators](../../01-creating-custom-evaluators)의 코드로 사용자 지정 evaluator를 생성해야 합니다.

### 학습 내용
- AgentCore Starter toolkit을 사용하여 trace에 온디맨드 평가를 실행하는 방법

### 튜토리얼 세부 정보

| 정보                | 세부 정보                                                                     |
|:--------------------|:------------------------------------------------------------------------------|
| 튜토리얼 유형       | 온디맨드 evaluator(기본 제공 및 사용자 지정)를 사용한 LangGraph 에이전트 평가 |
| 튜토리얼 구성 요소 | 기본 제공 및 사용자 지정 evaluator를 사용한 평가 실행                         |
| 튜토리얼 분야       | 산업군 공통                                                                    |
| 예제 난이도         | 쉬움                                                                           |
| 사용한 SDK          | Amazon Bedrock AgentCore Starter toolkit                                       |

### 온디맨드 평가

온디맨드 평가는 선택한 span 집합을 직접 분석하여 특정 에이전트 상호 작용을 유연하게 평가하는 방법을 제공합니다. 프로덕션 트래픽을 지속적으로 모니터링하는 온라인 평가와 달리, 온디맨드 평가를 사용하면 언제든지 선택한 상호 작용을 집중적으로 평가할 수 있습니다.

온디맨드 평가에서는 span, trace 또는 세션 ID를 제공하여 평가할 span, trace 또는 세션을 정확히 지정합니다. AgentCore Starter toolkit을 사용하면 세션의 모든 trace를 자동으로 평가할 수도 있습니다.

그런 다음 에이전트 상호 작용에 사용자 지정 evaluator 또는 기본 제공 evaluator를 적용할 수 있습니다. 이 평가 유형은 특정 고객 상호 작용을 조사하거나, 보고된 문제의 수정 사항을 검증하거나, 품질 개선을 위해 과거 데이터를 분석할 때 특히 유용합니다. 평가 요청을 제출하면 서비스가 지정된 span만 처리하고 분석을 위한 상세 결과를 제공합니다.

### 에이전트에서 AgentCore Observability trace 생성

AgentCore Observability는 상세한 실행 데이터를 캡처하고 구조화하는 기반으로 [OpenTelemetry (OTEL)](https://opentelemetry.io/) trace를 활용하여 호출 중 에이전트 동작을 포괄적으로 보여줍니다. AgentCore는 [AWS Distro for OpenTelemetry (ADOT)](https://aws-otel.github.io/)를 사용하여 다양한 에이전트 프레임워크에서 여러 유형의 OTEL trace를 계측합니다.

이 튜토리얼의 에이전트처럼 AgentCore Runtime에서 에이전트를 호스팅하면 최소한의 구성만으로 AgentCore Observability 계측이 자동 적용됩니다. `requirements.txt`에 `aws-opentelemetry-distro`를 포함하면 AgentCore Runtime이 OTEL 구성을 자동으로 처리합니다. 에이전트가 AgentCore Runtime에서 실행되지 않는 경우에는 AgentCore Observability에서 사용할 수 있도록 ADOT로 계측해야 합니다. telemetry 데이터를 CloudWatch로 전송하도록 환경 변수를 구성하고 OpenTelemetry 계측을 적용하여 에이전트를 실행해야 합니다.

과정은 다음과 같습니다.

![AgentCore Observability 세션 trace](../../images/observability_traces.png)

세션 trace를 AgentCore Observability에서 사용할 수 있게 되면 AgentCore Evaluations로 에이전트 동작을 평가할 수 있습니다.

### trace를 사용한 온디맨드 평가의 작동 방식

온디맨드 평가에서는 에이전트가 호출되어 AgentCore Observability에 trace를 생성합니다. 이러한 trace는 세션에 매핑되고 해당 로그는 Amazon CloudWatch log group에서 사용할 수 있습니다. 개발자는 평가에 사용할 세션이나 trace를 결정하고, trace 내용을 평가할 metric과 함께 AgentCore Evaluations의 입력으로 전송합니다. 과정은 다음과 같습니다.


![온디맨드 평가 흐름](../../images/on_demand_evaluations.png)

### 이전 튜토리얼의 정보 불러오기

이 튜토리얼에서는 사전 요구 사항 튜토리얼에서 AgentCore Runtime에 배포한 LangGraph 에이전트를 사용합니다. 기본 제공 metric과 `01-creating-custom-metrics` 튜토리얼에서 생성한 `response_quality` metric으로 에이전트를 평가합니다. 에이전트와 evaluator 정보를 불러오겠습니다.

In [ ]:
%store -r launch_result_langgraph
%store -r session_id_langgraph
%store -r evaluator_id
try:
    print("Agent Id:", launch_result_langgraph.agent_id)
    print("Agent ARN:", launch_result_langgraph.agent_arn)
except NameError:
    raise Exception(
        """Missing launch results from your LangGraph agent. Please run 00-prereqs before executing this lab"""
    )

try:
    print("Session id:", session_id_langgraph)
except NameError:
    raise Exception("""Missing session id from your LangGraph agent. Please run 00-prereqs before executing this lab""")

try:
    print("Evaluator id:", evaluator_id)
except NameError:
    raise Exception(
        """Missing custom evaluator id. Please run 01-creating-custom-evaluators before executing this lab"""
    )

### AgentCore Evaluations client 초기화

이제 AgentCore Starter toolkit에서 AgentCore Evaluations client를 초기화하겠습니다.

In [ ]:
from bedrock_agentcore_starter_toolkit import Evaluation
from boto3.session import Session
from IPython.display import Markdown, display

In [ ]:
boto_session = Session()
region = boto_session.region_name
print(region)

In [ ]:
eval_client = Evaluation(region=region)

### 평가 실행

AgentCore Evaluations를 실행하려면 세션, trace 또는 span 정보를 제공해야 합니다. 이전 튜토리얼에서 살펴본 것처럼 metric마다 에이전트 trace에서 요구하는 정보 수준이 다릅니다.

![metric 수준](../../images/metrics_per_level.png)

AWS SDK 중 하나를 사용하는 경우 trace를 직접 처리해야 합니다. AgentCore Starter toolkit은 세션 ID 또는 trace ID를 기반으로 trace를 처리하여 이 과정을 간소화합니다. 따라서 `run` 호출 시 AgentCore Starter toolkit evaluator client에 세션 ID를 제공하면 SDK가 해당 세션의 모든 trace를 추출하고 trace 및 span 수준 metric으로 평가합니다. 선택적으로 평가 결과를 저장할 출력 파일 이름도 제공할 수 있습니다.

### Goal Success Rate

이제 에이전트의 Goal Success Rate를 평가하겠습니다. 에이전트에 다음 질문을 했습니다.

* What is the weather now?
* How much is 2+2?
* Can you tell me the capital of the US?

In [ ]:
goal_sucess_results = eval_client.run(
    agent_id=launch_result_langgraph.agent_id,
    session_id=session_id_langgraph,
    evaluators=["Builtin.GoalSuccessRate"],
)

이제 evaluator 결과를 살펴보겠습니다. 결과 객체에는 수행된 평가 정보(`session_id`, `trace_id`, `input_data`)와 평가 결과가 포함됩니다.

평가 결과에는 evaluator 정보(id, name, ARN), 평가 값, 평가 label, 평가 설명과 평가 작업에 관한 추가 context(`spanContext`, `token_usage` 등)가 포함됩니다.

평가 응답을 확인해 보겠습니다.

In [ ]:
for result in goal_sucess_results.results:
    information = f"""
    Goal Success: {result.label} ({result.value})
    Explanation: \n{result.explanation}]\n
    Token Usage: {result.token_usage}\n
    Context: {result.context}\n
    """
    display(Markdown(information))

### Correctness

이제 동일한 세션을 trace 수준 metric인 Correctness로 분석하겠습니다.

In [ ]:
correctness_results = eval_client.run(
    agent_id=launch_result_langgraph.agent_id,
    session_id=session_id_langgraph,
    evaluators=["Builtin.Correctness"],
)

이제 evaluator 결과를 살펴보겠습니다. 이 경우 Correctness는 trace 수준에서 평가되므로 각 trace에 개별 평가 결과가 생성됩니다.

In [ ]:
for result in correctness_results.results:
    information = f"""
    Correctness: {result.label} ({result.value})
    Explanation: \n{result.explanation}]\n
    Token Usage: {result.token_usage}\n
    Context: {result.context}\n
    """
    display(Markdown(information))
    print("================================================")

### Tool selection accuracy and parameter selection accuracy

이제 에이전트의 tool 및 parameter 선택을 평가하겠습니다. 두 metric 모두 span 수준에서 평가됩니다.

In [ ]:
parameter_results = eval_client.run(
    agent_id=launch_result_langgraph.agent_id,
    session_id=session_id_langgraph,
    evaluators=["Builtin.ToolParameterAccuracy", "Builtin.ToolSelectionAccuracy"],
)

이제 결과를 분석하겠습니다. 여기서는 한 번의 실행으로 세션을 서로 다른 두 metric으로 평가하므로 각 응답을 생성한 evaluator를 식별해야 합니다. 결과의 `evaluator_name` 속성을 사용하면 이를 확인할 수 있습니다. 에이전트가 tool을 얼마나 잘 사용했는지 살펴보겠습니다.

In [ ]:
for result in parameter_results.results:
    information = f"""
    Metric: {result.evaluator_name}
    Value: {result.label} ({result.value})
    Explanation: \n{result.explanation}]\n
    Token Usage: {result.token_usage}\n
    Context: {result.context}\n
    """
    display(Markdown(information))
    print("================================================")

### 사용자 지정 evaluator 사용

세션, trace 및 span 수준 evaluator를 사용해 보았으므로 이제 사용자 지정 metric으로 응답 품질을 평가하겠습니다.

In [ ]:
custom_results = eval_client.run(
    agent_id=launch_result_langgraph.agent_id,
    session_id=session_id_langgraph,
    evaluators=[evaluator_id],
)

이제 평가 결과를 살펴보겠습니다. 여기서는 다음 지침이 설정된 에이전트를 평가합니다.

```
You're a helpful assistant. You can do simple math calculation, and tell the weather.
```

평가 지침에 명시된 대로, 평가 metric은 에이전트가 범위를 벗어나면 `Very Poor` 품질로 감점합니다.

```
...
**IMPORTANT**: A response quality can only be high if the agent remains in its original scope. Penalize agents that answer questions outside its original scope with a Very Poor classification.
...
```

다음 질문을 평가하므로,

* What is the weather now?
* How much is 2+2?
* Can you tell me the capital of the US?

마지막 질문에 대해 에이전트가 `Very Poor` 평가를 받을 것으로 예상합니다.

In [ ]:
for result in custom_results.results:
    information = f"""
    Metric: {result.evaluator_name}
    Value: {result.label} ({result.value})
    Explanation: \n{result.explanation}]\n
    Token Usage: {result.token_usage}\n
    Context: {result.context}\n
    """
    display(Markdown(information))
    print("================================================")

### 평가 결과 저장

AgentCore Starter toolkit을 사용하면 에이전트 평가 결과를 구조화된 출력 파일로 저장할 수도 있습니다. 실행할 때 `ouput` parameter를 제공하면 됩니다.

In [ ]:
save_results = eval_client.run(
    agent_id=launch_result_langgraph.agent_id,
    session_id=session_id_langgraph,
    evaluators=[evaluator_id],
    output="evals_results/output.json",
)

### 축하합니다!

이제 온디맨드 기능으로 에이전트를 평가했습니다. 다음 튜토리얼에서는 온라인 evaluator를 설정하고 에이전트에 연결하여 프로덕션 환경의 에이전트 평가를 자동화합니다.